In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import re
import os
import glob
import seaborn as sns
import io
import statistics

# 28S ExpansionSegments
data = np.array([
    ['ESL5', 'ESL7', 'ESL9', 'ESL10', 'ESL12', 'ESL15', 'ESL19', 'ESL20', 'ESL24', 'ESL26', 'ESL27', 'ESL30', 'ESL31', 'ESL39', 'ESL41'],
    [114, 465, 1384, 1682, 1793, 2075, 2439, 2538, 2685, 2751, 2875, 3955, 4057, 4698, 4983],
    [156, 1265, 1487, 1711, 1829, 2256, 2494, 2575, 2705, 2764, 3586, 4013, 4124, 4905, 4996]
])

df1 = pd.DataFrame(data.T, columns=['Segment', 'Start', 'End'])
df1['Start'] = pd.to_numeric(df1['Start']) + 7935
df1['End'] = pd.to_numeric(df1['End']) + 7935

# 18S ExpansionSegments
data = np.array([
    ['ESS1', 'ESS2', 'ESS3', 'ESS4', 'ESS6', 'ESS7', 'ESS8', 'ESS9', 'ESS10', 'ESS11', 'ESS12'],
    [53, 116, 208, 512, 737, 1097, 1272, 1398, 1533, 1563, 1726],
    [73, 130, 291, 583, 911, 1114, 1305, 1403, 1551, 1581, 1789]
])

df2 = pd.DataFrame(data.T, columns=['Segment', 'Start', 'End'])
df2['Start'] = pd.to_numeric(df2['Start']) + 3657
df2['End'] = pd.to_numeric(df2['End']) + 3657

df3 = pd.read_csv(r'/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/original reference/homopolymers.csv')

df4 = pd.read_csv(r'/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/original reference/Copy of conserve - Sheet1.csv')

df5 = pd.read_csv(r'/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/outputs/src_outputs/pg_positions_2.csv', sep=',', skipinitialspace=True, engine='python')

# for new vcf headers
def read_txt2(path, y_axis='AF_new'): #modified here to include y_axis parameter
    df = pd.read_csv(path)
    fxhold = df['POS'].tolist()
    refhold = df['new_ref'].tolist()
    althold = df['ALT_new'].tolist()
    fhold = df[y_axis].tolist() # normalized AF or AF_new
    return fhold, fxhold, refhold, althold

def get_rRNA_bounds(gb_file_path):
    gb_file = open(gb_file_path)
    pattern_1 = r'^\s+rRNA'
    pattern_2 = r'\d+'
    points = []
    for line in gb_file:
        if re.search(pattern_1, line):
            send = re.findall(pattern_2, line)
            points.append([int(send[0]), int(send[1])])
    return points

gb_file_path = '/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/original reference/U13369.1_Human_rDNA_repeat_unit.gb'
vcf_directory = '/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/outputs/src_outputs/new_vcf/full_set'
og_vcf_directory = '/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/original reference/original_vcf_all'

skipped_files_path = os.path.join(vcf_directory, 'skipped_files.csv')

if os.path.exists(skipped_files_path):
    skipped_files_df = pd.read_csv(skipped_files_path)
    skipped_files = set(skipped_files_df['Run'].apply(lambda x: f"new_vcf_{x}.csv"))
else:
    skipped_files = set()

if os.path.isdir(vcf_directory):
    vcf_files = [os.path.join(vcf_directory, file) for file in os.listdir(vcf_directory) if file.startswith('new') and file not in skipped_files]
else:
    raise ValueError(f"{vcf_directory} is not a valid directory")

# Get rRNA positions
rRNA_positions = get_rRNA_bounds(gb_file_path)
zoomed_range = [rRNA_positions[0][0] - 100, rRNA_positions[-1][-1] + 100]

expansion_positions1 = []
for row in df1.iterrows():
    hold = []
    hold.append(row[1]['Start'])
    hold.append(row[1]['End'])
    expansion_positions1.append(hold)

expansion_positions2 = []
for row in df2.iterrows():
    hold = []
    hold.append(row[1]['Start'])
    hold.append(row[1]['End'])
    expansion_positions2.append(hold)

hmplr_pos = []
for row in df3.iterrows():
    hold = []
    hold.append(row[1]['Start'])
    hold.append(row[1]['End'])
    hmplr_pos.append(hold)

csvd_pos = []
for row in df4.iterrows():
    hold = []
    hold.append(row[1]['Start'])
    hold.append(row[1]['End'])
    csvd_pos.append(hold)

In [2]:
start_28S, end_28S = 7935, 12969

expansion_segments = expansion_positions1
conserved_segments = csvd_pos  

# Modified here: Define set for pseudogene variants with position, ref, alt
pg_variants = set()
for _, row in df5.iterrows():
    start = row['Start']
    end = row['End']
    ref = row['Ref']
    alt = row['Alt']
    for pos in range(start, end + 1):
        pg_variants.add((pos, ref, alt))

hmplr_positions = set()
for _, row in df3.iterrows():
    start = row['Start']
    end = row['End']
    hmplr_positions.update(range(start, end + 1))

In [3]:
og_vcf_directory = '/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/original reference/original_vcf_all/'
modified_og_vcf_directory = '/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/outputs/src_outputs/modified_original_reference'

skipped_files_path = os.path.join(modified_og_vcf_directory, 'skipped_files.csv')

if os.path.exists(skipped_files_path):
    skipped_files_df = pd.read_csv(skipped_files_path)
    skipped_files = set(skipped_files_df['Run'].apply(lambda x: f"{x}_rDNA.vcf"))
else:
    skipped_files = set()

if os.path.isdir(og_vcf_directory):
    vcf_files = [os.path.join(og_vcf_directory, file) for file in os.listdir(og_vcf_directory) if file.startswith('E') and file not in skipped_files] # Modified here
else:
    raise ValueError(f"{og_vcf_directory} is not a valid directory")

file_indel_info = {}

for path in vcf_files:
    fn = os.path.basename(path)
    file_indel_info[fn] = []
    
    df = pd.read_csv(
        path,
        comment='#',
        sep='\t',
        header=None,
        names=['CHROM','POS','ID','REF','ALT','QUAL','FILTER','INFO']
    )
    
    for pos, ref, alt in zip(df.POS, df.REF, df.ALT):
        if (pos, ref, alt) in pg_variants:    continue
        if pos in hmplr_positions:            continue
        if pos > 13314:                       continue
        
        if len(ref) != len(alt):
            size = abs(len(ref) - len(alt))
            file_indel_info[fn].append({
                'pos':  pos,
                'ref':  ref,
                'alt':  alt,
                'size': size
            })

# find the largest indel per file
per_file_max = {
    fn: max(records, key=lambda r: r['size'])
    for fn, records in file_indel_info.items() if records
}

# overall largest
largest_fn, largest_rec = max(
    per_file_max.items(),
    key=lambda kv: kv[1]['size']
)
print(f"Largest indel: size={largest_rec['size']} at pos={largest_rec['pos']} in file={largest_fn}")

# top 5 across all files
all_recs = [
    {'file': fn, **rec}
    for fn, records in file_indel_info.items()
    for rec in records
]
df_all = pd.DataFrame(all_recs)
print("\nTop 5 indels overall:")
print(df_all.nlargest(5, 'size')[['file','pos','ref','alt','size']])

Largest indel: size=28 at pos=9312 in file=ERR3242735_rDNA.vcf

Top 5 indels overall:
                       file   pos                            ref alt  size
47522   ERR3242735_rDNA.vcf  9312  AAGGTGAAGGCCGGCGCGCTCGCCGGCCG   A    28
109916  ERR3989168_rDNA.vcf  9312  AAGGTGAAGGCCGGCGCGCTCGCCGGCCG   A    28
199299  ERR3239736_rDNA.vcf  8996  GCCACCCCTCCCACGGCGCGACCGCTCTC   G    28
18860   ERR3242583_rDNA.vcf  6298   GCGGCGGCCGTCGGGTGGGGGCTTTACC   G    27
23268   ERR3242044_rDNA.vcf  3290   GCGCCGGCCGCGCGCGCGCGCGCGTGGC   G    27


In [ ]:
import pandas as pd

vcf_file = '/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/original reference/original_vcf_all/ERR3242735_rDNA.vcf'

cols = ['CHROM','POS','ID','REF','ALT','QUAL','FILTER','INFO']

vcf_df = pd.read_csv(
    vcf_file,
    sep='\t',
    comment='#',
    header=None,
    names=cols
)

vcf_df[vcf_df.POS == 9312]

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
329,Human,9312,.,AAGGTGAAGGCCGGCGCGCTCGCCGGCCG,A,284,PASS,"DP=7360;AF=0.005299;SB=15;DP4=4547,2814,31,8;I..."
